In [1]:
# Clone the homework repository if it is not present
import os
import subprocess
repo_url = "https://github.com/mechristenson/aai-540-homework.git"
repo_path = "/home/ec2-user/SageMaker/MLOPSAssignments/aai-540-homework"
try:
    if os.path.isdir(os.path.join(repo_path, ".git")):
        print("Repository already exists.Skipping clone.")
    elif os.path.exists(repo_path):
        raise FileExistsError(
            f"Directory exists but is not a Git repository: {repo_path}"
        )
    else:
        subprocess.run(
            ["git", "clone", repo_url, repo_path],
            check=True
        )
        print("Repository cloned successfully.")
except Exception as e:
    print(f"Repository setup failed: {e}")
    raise

Repository already exists. Skipping clone.


In [2]:
#Imports, AWS configuration and dataset loading
import os
import time
import boto3
import pandas as pd

# AWS configuration
region = "us-east-1"
#Making the notebook reproducible
bucket_name = "aai540-homework-2-59870d8a"
database_name = "aai540_homework_2"
table_name = "tracks"
# Local dataset
file_path = (
    "/home/ec2-user/SageMaker/MLOPSAssignments/"
    "aai-540-homework/homework-2-1/data/dataset.csv"
)
try:
    # Create AWS clients
    s3 = boto3.client("s3", region_name=region)
    athena = boto3.client("athena", region_name=region)
    glue = boto3.client("glue", region_name=region)
    # Load dataset
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Dataset not found: {file_path}")
    df = pd.read_csv(file_path)
    print("AWS clients initialized successfully.")
    print("Dataset loaded successfully.")
    print("Dataset shape:", df.shape)
    print("Number of columns:", len(df.columns))
except Exception as e:
    print(f"Initialization failed: {e}")
    raise

AWS clients initialized successfully.
Dataset loaded successfully.
Dataset shape: (114000, 21)
Number of columns: 21


In [3]:
#Create S3 bucket and upload dataset
try:
    # Check whether the bucket already exists
    try:
        s3.head_bucket(Bucket=bucket_name)
        print(f"Bucket already exists: {bucket_name}")
    except s3.exceptions.ClientError:
        print(f"Bucket does not exist. Creating: {bucket_name}")
        if region == "us-east-1":
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={
                    "LocationConstraint": region
                }
            )
        print("Bucket created successfully.")
    # S3 location for the dataset
    s3_key = "homework-2-1/dataset.csv"
    # Check whether the object already exists
    try:
        existing_object = s3.head_object(
            Bucket=bucket_name,
            Key=s3_key
        )
        local_size = os.path.getsize(file_path)
        if existing_object["ContentLength"] == local_size:
            print("Dataset already exists in S3. Upload skipped.")
        else:
            s3.upload_file(file_path, bucket_name, s3_key)
            print("Dataset updated in S3.")
    except s3.exceptions.ClientError:
        s3.upload_file(file_path, bucket_name, s3_key)
        print("Dataset uploaded successfully.")
    print(f"S3 location: s3://{bucket_name}/{s3_key}")
except Exception as e:
    print(f"S3 setup failed: {e}")
    raise

Bucket already exists: aai540-homework-2-59870d8a
Dataset already exists in S3. Upload skipped.
S3 location: s3://aai540-homework-2-59870d8a/homework-2-1/dataset.csv


In [4]:
#Create Athena database if it does not exist
try:
    database_query = f"""
    CREATE DATABASE IF NOT EXISTS {database_name}
    """
    response = athena.start_query_execution(
        QueryString=database_query,
        ResultConfiguration={
            "OutputLocation": f"s3://{bucket_name}/athena-results/"
        }
    )
    query_execution_id = response["QueryExecutionId"]
    # Wait for Athena to finish
    while True:
        status = athena.get_query_execution(
            QueryExecutionId=query_execution_id
        )["QueryExecution"]["Status"]
        state = status["State"]
        if state == "SUCCEEDED":
            print(f"Athena database ready: {database_name}")
            break
        if state in ["FAILED", "CANCELLED"]:
            raise RuntimeError(
                f"Athena database creation {state}: "
                f"{status.get('StateChangeReason', 'Unknown error')}"
            )
        time.sleep(2)
except Exception as e:
    print(f"Athena database setup failed: {e}")
    raise

Athena database ready: aai540_homework_2


In [5]:
#Create the Athena/Glue external table if necessary
columns = [
    {"Name": "unnamed_0", "Type": "bigint"},
    {"Name": "track_id", "Type": "string"},
    {"Name": "artists", "Type": "string"},
    {"Name": "album_name", "Type": "string"},
    {"Name": "track_name", "Type": "string"},
    {"Name": "popularity", "Type": "bigint"},
    {"Name": "duration_ms", "Type": "bigint"},
    {"Name": "explicit", "Type": "boolean"},
    {"Name": "danceability", "Type": "double"},
    {"Name": "energy", "Type": "double"},
    {"Name": "track_key", "Type": "bigint"},
    {"Name": "loudness", "Type": "double"},
    {"Name": "mode", "Type": "bigint"},
    {"Name": "speechiness", "Type": "double"},
    {"Name": "acousticness", "Type": "double"},
    {"Name": "instrumentalness", "Type": "double"},
    {"Name": "liveness", "Type": "double"},
    {"Name": "valence", "Type": "double"},
    {"Name": "tempo", "Type": "double"},
    {"Name": "time_signature", "Type": "bigint"},
    {"Name": "track_genre", "Type": "string"}
]
try:
    # Check whether the Glue table already exists
    try:
        glue.get_table(
            DatabaseName=database_name,
            Name=table_name
        )
        print(
            f"Glue/Athena table already exists: "
            f"{database_name}.{table_name}"
        )
    except glue.exceptions.EntityNotFoundException:
        table_input = {
            "Name": table_name,
            "TableType": "EXTERNAL_TABLE",
            "Parameters": {
                "classification": "csv",
                "skip.header.line.count": "1",
                "typeOfData": "file"
            },
            "StorageDescriptor": {
                "Columns": columns,
                "Location": (
                    f"s3://{bucket_name}/homework-2-1/"
                ),
                "InputFormat": (
                    "org.apache.hadoop.mapred.TextInputFormat"
                ),
                "OutputFormat": (
                    "org.apache.hadoop.hive.ql.io."
                    "HiveIgnoreKeyTextOutputFormat"
                ),
                "SerdeInfo": {
                    "SerializationLibrary": (
                        "org.apache.hadoop.hive.serde2.OpenCSVSerde"
                    ),
                    "Parameters": {
                        "separatorChar": ",",
                        "quoteChar": "\"",
                        "escapeChar": "\\"
                    }
                }
            }
        }
        glue.create_table(
            DatabaseName=database_name,
            TableInput=table_input
        )
        print("Glue/Athena table created successfully.")
except Exception as e:
    print(f"Table setup failed: {e}")
    raise

Glue/Athena table already exists: aai540_homework_2.tracks


In [6]:
#Ensure that  AWS Data Wrangler is installed
try:
    import awswrangler as wr
    print("AWS Wrangler already installed.")
    print("Version:", wr.__version__)
except ImportError:
    print("AWS Wrangler not found. Installing...")
    import subprocess
    import sys
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "awswrangler",
            "-q"
        ],
        check=True
    )
    import awswrangler as wr
    print("AWS Wrangler installed successfully.")
    print("Version:", wr.__version__)

AWS Wrangler already installed.
Version: 3.17.1


In [7]:
#Validate the complete S3 -> Glue -> Athena pipeline
s3_output = f"s3://{bucket_name}/athena-results/"
try:
    validation_query = f"""
    SELECT COUNT(*) AS row_count
    FROM {database_name}.{table_name}
    """
    validation_result = wr.athena.read_sql_query(
        sql=validation_query,
        database=database_name,
        s3_output=s3_output,
        ctas_approach=False
    )
    row_count = validation_result.loc[0, "row_count"]
    print("Athena validation successful.")
    print("Rows available:", row_count)
    if row_count != len(df):
        raise ValueError(
            f"Row count mismatch: "
            f"Athena={row_count}, Local={len(df)}"
        )
    print("S3 dataset verified")
    print("Glue table verified")
    print("Athena query verified")
    print("Row counts match")
except Exception as e:
    print(f"Athena validation failed: {e}")
    raise

Athena validation successful.
Rows available: 114000
S3 dataset verified
Glue table verified
Athena query verified
Row counts match


In [11]:
def run_athena_query(query_name: str, sql_query: str):
    """
    Execute an Athena SQL query and return the result as a Pandas DataFrame.
    """
    try:
        result = wr.athena.read_sql_query(
            sql=sql_query,
            database=database_name,
            s3_output=s3_output,
            ctas_approach=False
        )
        print(f"{query_name} executed successfully.")
        print(f"Returned rows: {len(result)}")
        display(result)
        return result
    except Exception as e:
        print(f"{query_name} failed: {e}")
        raise


def validate_query_result(
    query_name: str,
    athena_result,
    expected_result,
    sort_columns=None
):
    """
    Validate an Athena result against an independently calculated
    expected result from the original Pandas dataset.
    The validation checks:
    - Same number of rows
    - Same columns
    - Same values
    - No unexpected/missing rows
     the expected result is calculated directly from the source dataset.
    """
    try:
        actual = athena_result.copy()
        expected = expected_result.copy()
        # Normalize column names
        actual.columns = actual.columns.str.lower()
        expected.columns = expected.columns.str.lower()
        # Check columns
        if set(actual.columns) != set(expected.columns):
            raise AssertionError(
                f"Column mismatch.\n"
                f"Athena columns: {list(actual.columns)}\n"
                f"Expected columns: {list(expected.columns)}"
            )
        # Put columns in the same order
        expected = expected[actual.columns]
        # Sort both results so row ordering does not affect validation
        if sort_columns is None:
            sort_columns = list(actual.columns)
        actual = actual.sort_values(
            by=sort_columns
        ).reset_index(drop=True)
        expected = expected.sort_values(
            by=sort_columns
        ).reset_index(drop=True)
        # Compare row counts
        if len(actual) != len(expected):
            raise AssertionError(
                f"Row count mismatch.\n"
                f"Athena returned: {len(actual)}\n"
                f"Expected: {len(expected)}"
            )
        # Compare values
        pd.testing.assert_frame_equal(
            actual,
            expected,
            check_dtype=False,
            check_exact=False,
            rtol=1e-10,
            atol=1e-10
        )
        print(f"{query_name} validation PASSED.")
        print("Athena result matches the independently calculated")
        print("result from the original dataset.")
        print(f"Validated rows: {len(actual)}")
        return True
    except Exception as e:
        print(f"{query_name} validation FAILED.")
        print(f"Reason: {e}")
        raise

In [12]:
# Define the 5 Assignment Queries
homework_queries = {
    "Query 1": """
        SELECT DISTINCT
            artists,
            track_name,
            popularity
        FROM aai540_homework_2.tracks
        WHERE popularity >= 99
        ORDER BY popularity DESC, artists, track_name
    """,
    "Query 2": """
        SELECT
            artists,
            AVG(popularity) AS avg_popularity
        FROM aai540_homework_2.tracks
        GROUP BY artists
        HAVING AVG(popularity) = 92
        ORDER BY artists
    """,
    "Query 3": """
        SELECT
            track_genre,
            AVG(energy) AS avg_energy
        FROM aai540_homework_2.tracks
        GROUP BY track_genre
        ORDER BY avg_energy DESC
        LIMIT 10
    """,
    "Query 4": """
        SELECT
            COUNT(*) AS track_count
        FROM aai540_homework_2.tracks
        WHERE LOWER(artists) LIKE '%bad bunny%'
    """,
    "Query 5": """
        SELECT
            track_genre,
            MAX(popularity) AS most_popular_track
        FROM aai540_homework_2.tracks
        GROUP BY track_genre
        ORDER BY most_popular_track DESC, track_genre
        LIMIT 10
    """
}
print(f"Defined {len(homework_queries)} assignment queries.")
for query_name in homework_queries:
    print(f"- {query_name}")

Defined 5 assignment queries.
- Query 1
- Query 2
- Query 3
- Query 4
- Query 5


In [13]:
#Calculate Expected Results from Original Dataset
try:
    # Query 1
    expected_query_1 = (
        df.loc[
            df["popularity"] >= 99,
            ["artists", "track_name", "popularity"]
        ]
        .drop_duplicates()
        .sort_values(
            ["popularity", "artists", "track_name"],
            ascending=[False, True, True]
        )
        .reset_index(drop=True)
    )
    # Query 2
    expected_query_2 = (
        df.groupby("artists", as_index=False)["popularity"]
        .mean()
        .rename(columns={"popularity": "avg_popularity"})
    )
    expected_query_2 = (
        expected_query_2[
            expected_query_2["avg_popularity"] == 92
        ]
        .sort_values("artists")
        .reset_index(drop=True)
    )
    # Query 3
    expected_query_3 = (
        df.groupby("track_genre", as_index=False)["energy"]
        .mean()
        .rename(columns={"energy": "avg_energy"})
        .sort_values(
            ["avg_energy", "track_genre"],
            ascending=[False, True]
        )
        .head(10)
        .reset_index(drop=True)
    )
    # Query 4
    bad_bunny_mask = (
        df["artists"]
        .fillna("")
        .str.lower()
        .str.contains("bad bunny", regex=False)
    )
    expected_query_4 = pd.DataFrame({
        "track_count": [bad_bunny_mask.sum()]
    })
    # Query 5
    expected_query_5 = (
        df.groupby("track_genre", as_index=False)["popularity"]
        .max()
        .rename(columns={"popularity": "most_popular_track"})
        .sort_values(
            ["most_popular_track", "track_genre"],
            ascending=[False, True]
        )
        .head(10)
        .reset_index(drop=True)
    )
    expected_results = {
        "Query 1": expected_query_1,
        "Query 2": expected_query_2,
        "Query 3": expected_query_3,
        "Query 4": expected_query_4,
        "Query 5": expected_query_5
    }
    print("Expected results calculated successfully from the original dataset.")
except Exception as e:
    print(f"Expected-result calculation failed: {e}")
    raise

Expected results calculated successfully from the original dataset.


In [14]:
#Execute and validate allAssignment queries
query_results = {}
validation_results = {}
for query_name, sql_query in homework_queries.items():
    print(query_name)
    # Execute Athena query
    result = run_athena_query(
        query_name=query_name,
        sql_query=sql_query
    )
    query_results[query_name] = result
    # Validate against independently calculated Pandas result
    validation_results[query_name] = validate_query_result(
        query_name=query_name,
        athena_result=result,
        expected_result=expected_results[query_name]
    )
print("Final validation summary!!!")
for query_name, passed in validation_results.items():
    print(f"{query_name}: {'PASSED' if passed else 'FAILED'}")

Query 1
Query 1 executed successfully.
Returned rows: 2


,artists,track_name,popularity
0,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),100
1,Bizarrap;Quevedo,"Quevedo: Bzrp Music Sessions, Vol. 52",99


Query 1 validation PASSED.
Athena result matches the independently calculated
result from the original dataset.
Validated rows: 2
Query 2
Query 2 executed successfully.
Returned rows: 2


,artists,avg_popularity
0,Harry Styles,92.0
1,Rema;Selena Gomez,92.0


Query 2 validation PASSED.
Athena result matches the independently calculated
result from the original dataset.
Validated rows: 2
Query 3
Query 3 executed successfully.
Returned rows: 10


...

Query 3 validation PASSED.
Athena result matches the independently calculated
result from the original dataset.
Validated rows: 10
Query 4
Query 4 executed successfully.
Returned rows: 1


,track_count
0,416


Query 4 validation PASSED.
Athena result matches the independently calculated
result from the original dataset.
Validated rows: 1
Query 5
Query 5 executed successfully.
Returned rows: 10


,track_genre,most_popular_track
0,dance,100
1,pop,100
2,hip-hop,99
3,edm,98
4,latin,98
5,latino,98
6,reggae,98
7,reggaeton,98
8,piano,96
9,rock,96


Query 5 validation PASSED.
Athena result matches the independently calculated
result from the original dataset.
Validated rows: 10
Final validation summary!!!
Query 1: PASSED
Query 2: PASSED
Query 3: PASSED
Query 4: PASSED
Query 5: PASSED


In [15]:
# Create week2 directory, copy notebook, commit, and push
import os
import shutil
import subprocess
repo = "/home/ec2-user/SageMaker/MLOPSAssignments"
source = "/home/ec2-user/SageMaker/MLOPSAssignments/aai-540-homework/untitled.ipynb"
destination_dir = os.path.join(repo, "week2")
destination = os.path.join(destination_dir, "untitled.ipynb")
try:
    # Create week2 directory
    os.makedirs(destination_dir, exist_ok=True)
    # Copy notebook
    shutil.copy2(source, destination)
    # Git add, commit, and push
    subprocess.run(["git", "add", "week2"], cwd=repo, check=True)
    subprocess.run(
        ["git", "commit", "-m", "Add Week 2 homework"],
        cwd=repo,
        check=True
    )
    subprocess.run(["git", "push", "origin", "main"], cwd=repo, check=True)

    print("Week 2 notebook pushed successfully!")
    print(f"{destination}")
except Exception as e:
    print(f"Failed: {e}")

Failed: [Errno 2] No such file or directory: '/home/ec2-user/SageMaker/MLOPSAssignments/aai-540-homework/untitled.ipynb'
